<a href="https://colab.research.google.com/github/lakeontario1990/colab_notebooks/blob/main/Untitled47.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title PCBS sector value-added calculation from raw quarterly data, 2023-2025
# Purpose:
# This cell hardcodes the quarterly PCBS values shown in your screenshots, then calculates:
# 1) 2025 annual level = Q1+Q2+Q3+Q4
# 2) % change vs 2023 annual level
# 3) % change vs 2024 annual level
# 4) Q4 2025 y/y = Q4 2025 / Q4 2024 - 1
#
# Important:
# - No PCBS rows are removed.
# - Aggregate rows such as "Services", "Mining, Manufacturing, Electricity and Water",
#   "Customs duties", "VAT on imports, net", and "Gross Domestic Product" are kept.
# - Do not sum the final table without filtering, because it contains both aggregates
#   and their components.
# - 2023 is taken from the 2023/2024 PCBS screenshot.
# - 2024 and 2025 are taken from the later 2024/2025 PCBS screenshot, because the 2024
#   figures in the earlier screenshot were later revised.

!pip -q install pandas openpyxl xlsxwriter

from IPython.display import display, clear_output
clear_output()

import os
import numpy as np
import pandas as pd

OUTDIR = "/content/pcbs_sector_calc_checkpoints"
os.makedirs(OUTDIR, exist_ok=True)

# Values are stored as [Q1, Q2, Q3, Q4], even though the PCBS table itself is visually ordered right-to-left.
# Unit: USD million, constant 2015 prices.

rows = [
    ("Agriculture, Forestry and Fishing",
     [204.4, 250.9, 225.3, 154.6],
     [121.8, 157.8, 145.0, 157.7],
     [155.9, 159.1, 156.0, 151.8]),

    ("Mining, Manufacturing, Electricity and Water",
     [445.9, 463.0, 489.2, 312.1],
     [428.3, 465.2, 485.3, 496.3],
     [464.6, 502.8, 562.0, 525.3]),

    ("Mining and Quarrying",
     [12.6, 13.6, 13.5, 10.3],
     [16.2, 17.2, 16.3, 16.7],
     [16.0, 16.4, 16.0, 16.1]),

    ("Manufacturing",
     [385.9, 402.8, 422.4, 279.5],
     [391.0, 429.9, 448.8, 460.9],
     [429.8, 469.0, 524.7, 490.5]),

    ("Electricity, Gas, Steam and Air Conditioning Supply",
     [35.0, 34.8, 42.8, 14.7],
     [11.7, 10.6, 13.0, 11.4],
     [12.4, 11.6, 13.4, 11.9]),

    ("Water Supply, Sewerage, Waste Management and Remediation Activities",
     [12.4, 11.8, 10.5, 7.6],
     [9.4, 7.5, 7.2, 7.3],
     [6.4, 5.8, 7.9, 6.8]),

    ("Construction",
     [141.3, 131.5, 138.7, 86.6],
     [115.5, 113.2, 119.5, 130.0],
     [108.7, 137.3, 124.4, 117.0]),

    ("Wholesale and Retail Trade, Repair of Motor Vehicles and Motorcycles",
     [727.1, 717.3, 689.8, 521.8],
     [498.5, 551.2, 576.9, 578.5],
     [584.5, 583.8, 566.5, 574.8]),

    ("Transportation and Storage",
     [56.8, 55.2, 55.0, 37.0],
     [17.6, 20.5, 21.5, 20.2],
     [16.8, 19.9, 20.9, 17.9]),

    ("Financial and Insurance Activities",
     [202.7, 208.3, 211.8, 164.5],
     [169.4, 171.8, 171.7, 171.4],
     [169.4, 177.3, 168.1, 163.6]),

    ("Information and Communication",
     [111.4, 116.5, 120.4, 96.2],
     [90.4, 95.9, 98.6, 94.9],
     [95.9, 102.2, 106.7, 106.3]),

    ("Services",
     [813.9, 816.2, 823.2, 608.5],
     [606.1, 608.1, 619.6, 614.7],
     [672.2, 626.0, 609.9, 681.7]),

    ("Accommodation and Food Service Activities",
     [41.5, 46.2, 43.6, 27.8],
     [20.5, 22.2, 23.8, 23.7],
     [26.3, 27.8, 31.8, 31.9]),

    ("Real Estate Activities",
     [166.6, 153.0, 163.9, 94.1],
     [147.2, 144.1, 152.2, 165.6],
     [138.3, 174.8, 158.3, 148.9]),

    ("Professional, Scientific and Technical Activities",
     [41.4, 32.7, 41.2, 28.3],
     [23.9, 23.6, 26.5, 12.5],
     [18.3, 20.6, 21.5, 22.6]),

    ("Administrative and Support Service Activities",
     [23.3, 23.1, 19.1, 17.7],
     [15.4, 14.5, 9.6, 11.7],
     [15.4, 15.3, 7.6, 11.7]),

    ("Education",
     [251.6, 265.3, 256.9, 217.4],
     [209.3, 207.3, 207.1, 208.8],
     [241.6, 198.8, 207.0, 241.2]),

    ("Human Health and Social Work Activities",
     [193.7, 186.8, 191.8, 155.4],
     [155.2, 150.4, 155.9, 150.9],
     [181.5, 135.6, 139.5, 185.1]),

    ("Arts, Entertainment and Recreation",
     [15.7, 24.2, 14.9, 12.1],
     [4.3, 5.6, 6.7, 4.9],
     [3.1, 4.8, 5.3, 3.8]),

    ("Other Service Activities",
     [80.1, 84.9, 91.8, 55.7],
     [30.3, 40.4, 37.8, 36.6],
     [47.7, 48.3, 38.9, 36.5]),

    ("Public administration and defense",
     [449.9, 472.4, 445.2, 304.7],
     [223.7, 235.3, 231.7, 251.4],
     [230.2, 217.6, 222.1, 256.6]),

    ("Households with Employed Persons",
     [1.8, 1.8, 1.9, 1.7],
     [1.4, 1.4, 1.4, 1.2],
     [1.4, 1.4, 1.4, 1.4]),

    ("Customs duties",
     [424.6, 375.3, 437.9, 279.5],
     [262.5, 240.5, 258.4, 284.3],
     [270.4, 231.0, 271.8, 296.6]),

    ("VAT on imports, net",
     [397.4, 399.4, 422.3, 309.8],
     [288.5, 288.8, 328.4, 314.0],
     [333.4, 284.6, 291.6, 322.1]),

    ("Gross Domestic Product",
     [3977.2, 4007.8, 4060.7, 2877.0],
     [2823.7, 2947.7, 3058.0, 3114.6],
     [3103.4, 3043.0, 3101.4, 3215.1]),
]

records = []
for input_order, (activity, y2023, y2024, y2025) in enumerate(rows, start=1):
    for year, values in [(2023, y2023), (2024, y2024), (2025, y2025)]:
        if len(values) != 4:
            raise ValueError(f"{activity} / {year} does not have exactly four quarterly values.")
        for quarter, value in zip(["Q1", "Q2", "Q3", "Q4"], values):
            records.append({
                "Input order": input_order,
                "Economic Activity": activity,
                "Year": year,
                "Quarter": quarter,
                "Value_USD_m_constant_2015": float(value)
            })

raw = pd.DataFrame(records)

# Basic input checks.
expected_rows = len(rows) * 3 * 4
if len(raw) != expected_rows:
    raise ValueError(f"Raw dataset has {len(raw)} rows, expected {expected_rows}.")

if raw["Value_USD_m_constant_2015"].isna().any():
    raise ValueError("Raw dataset contains missing values.")

q_count = raw.groupby(["Economic Activity", "Year"])["Quarter"].nunique()
if not (q_count == 4).all():
    raise ValueError("At least one activity-year does not have all four quarters.")

# Annual calculations.
annual = (
    raw
    .pivot_table(
        index=["Input order", "Economic Activity"],
        columns="Year",
        values="Value_USD_m_constant_2015",
        aggfunc="sum"
    )
    .reset_index()
)

annual.columns.name = None
annual = annual.rename(columns={
    2023: "2023 annual level, USD m",
    2024: "2024 annual level, USD m",
    2025: "2025 GDP level (USD m)"
})

q4 = (
    raw[raw["Quarter"] == "Q4"]
    .pivot_table(
        index=["Economic Activity"],
        columns="Year",
        values="Value_USD_m_constant_2015",
        aggfunc="sum"
    )
    .reset_index()
    .rename(columns={2024: "Q4 2024", 2025: "Q4 2025"})
)

out = annual.merge(q4[["Economic Activity", "Q4 2024", "Q4 2025"]], on="Economic Activity", how="left")

out["% change vs 2023"] = (out["2025 GDP level (USD m)"] / out["2023 annual level, USD m"] - 1) * 100
out["% change vs 2024"] = (out["2025 GDP level (USD m)"] / out["2024 annual level, USD m"] - 1) * 100
out["Q4 2025 y/y (%)"] = (out["Q4 2025"] / out["Q4 2024"] - 1) * 100
out["Rank"] = out["2025 GDP level (USD m)"].rank(method="min", ascending=False).astype(int)

final = (
    out[[
        "Rank",
        "Economic Activity",
        "2025 GDP level (USD m)",
        "% change vs 2023",
        "% change vs 2024",
        "Q4 2025 y/y (%)",
        "Input order",
        "2023 annual level, USD m",
        "2024 annual level, USD m",
        "Q4 2024",
        "Q4 2025",
    ]]
    .sort_values(["Rank", "Input order"])
    .reset_index(drop=True)
)

# Rounded display version.
display_table = final.copy()
for col in [
    "2025 GDP level (USD m)",
    "% change vs 2023",
    "% change vs 2024",
    "Q4 2025 y/y (%)",
    "2023 annual level, USD m",
    "2024 annual level, USD m",
    "Q4 2024",
    "Q4 2025",
]:
    display_table[col] = display_table[col].round(1)

# Save checkpoints.
raw_csv = os.path.join(OUTDIR, "pcbs_raw_quarterly_values_2023_2025.csv")
final_csv = os.path.join(OUTDIR, "pcbs_sector_value_added_calculated_table_2025.csv")
xlsx_path = os.path.join(OUTDIR, "pcbs_sector_value_added_calculation_2025.xlsx")

raw.to_csv(raw_csv, index=False)
final.to_csv(final_csv, index=False)

with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
    raw.to_excel(writer, sheet_name="raw_quarterly_values", index=False)
    final.to_excel(writer, sheet_name="calculated_table", index=False)

print("Done. Checkpoints saved:")
print(f"- {raw_csv}")
print(f"- {final_csv}")
print(f"- {xlsx_path}")
print()
print("Full calculated table, all PCBS rows retained:")
display(display_table)

Done. Checkpoints saved:
- /content/pcbs_sector_calc_checkpoints/pcbs_raw_quarterly_values_2023_2025.csv
- /content/pcbs_sector_calc_checkpoints/pcbs_sector_value_added_calculated_table_2025.csv
- /content/pcbs_sector_calc_checkpoints/pcbs_sector_value_added_calculation_2025.xlsx

Full calculated table, all PCBS rows retained:


,Rank,Economic Activity,2025 GDP level (USD m),% change vs 2023,% change vs 2024,Q4 2025 y/y (%),Input order,"2023 annual level, USD m","2024 annual level, USD m",Q4 2024,Q4 2025
0,1,Gross Domestic Product,12462.9,-16.5,4.3,3.2,25,14922.7,11944.0,3114.6,3215.1
1,2,Services,2589.8,-15.4,5.8,10.9,12,3061.8,2448.5,614.7,681.7
2,3,"Wholesale and Retail Trade, Repair of Motor Ve...",2309.6,-13.0,4.7,-0.6,8,2656.0,2205.1,578.5,574.8
3,4,"Mining, Manufacturing, Electricity and Water",2054.7,20.1,9.6,5.8,2,1710.2,1875.1,496.3,525.3
4,5,Manufacturing,1914.0,28.4,10.6,6.4,4,1490.6,1730.6,460.9,490.5
5,6,"VAT on imports, net",1231.7,-19.4,1.0,2.6,24,1528.9,1219.7,314.0,322.1
6,7,Customs duties,1069.8,-29.5,2.3,4.3,23,1517.3,1045.7,284.3,296.6
7,8,Public administration and defense,926.5,-44.6,-1.7,2.1,21,1672.2,942.1,251.4,256.6
8,9,Education,888.6,-10.4,6.7,15.5,17,991.2,832.5,208.8,241.2
9,10,Financial and Insurance Activities,678.4,-13.8,-0.9,-4.6,10,787.3,684.3,171.4,163.6
